# Section 1: Environment Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/ECE1508_StyleID'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/outputs', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/results', exist_ok=True)

print("Project directory structure:")
for root, dirs, files in os.walk(PROJECT_ROOT):
    level = root.replace(PROJECT_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')

Project directory structure:
ECE1508_StyleID/
  checkpoints/
  data/
    annotations/
    content/
    style/
  outputs/
  results/


In [3]:
!pip install --upgrade diffusers==0.38.0 transformers accelerate peft huggingface_hub -q
!pip install pycocotools -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 145.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 165.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 65.9 MB/s eta 0:00:00


In [4]:
import torch
import diffusers
import transformers

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"diffusers version: {diffusers.__version__}")
print(f"transformers version: {transformers.__version__}")

PyTorch version: 2.11.0+cu128
CUDA available: True
diffusers version: 0.38.0
transformers version: 5.14.1


In [5]:
from diffusers import AutoencoderKL, UNet2DConditionModel, DDIMScheduler
from transformers import CLIPTextModel, CLIPTokenizer
import torch

model_id = "runwayml/stable-diffusion-v1-5"
device = "cuda"
dtype = torch.float16

vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae", torch_dtype=dtype).to(device)
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet", torch_dtype=dtype).to(device)
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder", torch_dtype=dtype).to(device)
scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler")

vae.eval()
unet.eval()
text_encoder.eval()

empty_input = tokenizer([""], padding="max_length", max_length=tokenizer.model_max_length, return_tensors="pt")
text_embeddings = text_encoder(empty_input.input_ids.to(device))[0]

print("All components loaded")
print(f"UNet has {sum(p.numel() for p in unet.parameters())/1e6:.1f}M parameters")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B /  335MB            

vae/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

text_encoder/model.safetensors: reconstructing file:   0%|          |  0.00B /  492MB            

text_encoder/model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

All components loaded
UNet has 859.5M parameters
